# Ligand preparation for AutoDock Vina

This notebook converts ligands from SDF and SMILES formats into PDBQT format for docking with AutoDock Vina

Pipeline:
- Read input molecules (SDF and SMI)
- Add hydrogens and generate 3D coordinates if needed
- Prepare molecules using Meeko
- Export to PDBQT format

## Imports

In [ ]:
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import AllChem

from meeko import MoleculePreparation, PDBQTWriterLegacy

## Configuration

In [ ]:
INPUT_SDF = Path("input/example.sdf")
INPUT_SMI = Path("input/example.smi")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Functions

In [ ]:
def prepare_molecule_rdkit(mol):
    mol = Chem.Mol(mol)
    mol = Chem.AddHs(mol)

    is_3d = False
    if mol.GetNumConformers() > 0:
        is_3d = mol.GetConformer().Is3D()

    if not is_3d:
        params = AllChem.ETKDGv3()
        params.randomSeed = 42

        status = AllChem.EmbedMolecule(mol, params)
        if status != 0:
            raise ValueError("Failed to generate 3D conformer")

        AllChem.UFFOptimizeMolecule(mol)

    return mol


def write_pdbqt(mol, output_path, flexible_macrocycles=True):
    """Prepare molecule with Meeko and write to PDBQT file.
    
    Aliphatic rings with 6+ atoms are treated as flexible by default.
    """
    preparator = MoleculePreparation(
        rigid_macrocycles=not flexible_macrocycles,
        min_ring_size=6,
    )
    setups = preparator.prepare(mol)

    if not setups:
        raise ValueError("Meeko failed to prepare molecule")

    pdbqt_string, is_ok, error_msg = PDBQTWriterLegacy.write_string(setups[0])

    if not is_ok:
        raise ValueError(error_msg)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(pdbqt_string)

## Load data

Quick exploratory look at the input files before processing

In [ ]:
# SDF overview
supplier = Chem.SDMolSupplier(str(INPUT_SDF))
print("Molecules in SDF:", len(supplier))

for i, mol in enumerate(supplier):
    if mol is None:
        print(f"  [{i}] parse error")
        continue

    name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"ligand_{i}"
    conformers = mol.GetNumConformers()
    is_3d = mol.GetConformer().Is3D() if conformers > 0 else False
    print(f"  [{i}] {name} | atoms: {mol.GetNumAtoms()} | conformers: {conformers} | 3D: {is_3d}")

In [ ]:
# SMI overview — first 5 lines
with open(INPUT_SMI, "r") as f:
    for _ in range(5):
        print(f.readline().strip())

## Process SDF ligands

In [ ]:
supplier = Chem.SDMolSupplier(str(INPUT_SDF), removeHs=False)

sdf_success = 0
sdf_failed = 0

for i, mol in enumerate(supplier, start=1):
    if mol is None:
        print(f"[{i}] parse error")
        sdf_failed += 1
        continue

    name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"ligand_{i}"

    try:
        prepared_mol = prepare_molecule_rdkit(mol)
        write_pdbqt(prepared_mol, OUTPUT_DIR / f"{name}.pdbqt")
        sdf_success += 1
        print(f"Saved: {name}.pdbqt")

    except Exception as e:
        sdf_failed += 1
        print(f"Error for {name}: {e}")

print("---")
print("SDF success:", sdf_success)
print("SDF failed: ", sdf_failed)

## Process SMI ligands

SMILES input does not contain 3D coordinates, so 3D conformers are generated using RDKit before conversion to PDBQT

In [ ]:
smi_success = 0
smi_failed = 0

with open(INPUT_SMI, "r") as f:
    next(f)  # skip header: "smiles name"

    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        smiles = parts[0]
        name = parts[1] if len(parts) > 1 else f"ligand_{i}"

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Invalid SMILES [{i}]: {smiles}")
            smi_failed += 1
            continue

        try:
            prepared_mol = prepare_molecule_rdkit(mol)
            write_pdbqt(prepared_mol, OUTPUT_DIR / f"smi_{name}.pdbqt")
            smi_success += 1

        except Exception as e:
            smi_failed += 1
            print(f"Error for {name}: {e}")

print("---")
print("SMI success:", smi_success)
print("SMI failed: ", smi_failed)

## Summary

In [ ]:
print(f"Total — success: {total_success}, failed: {total_failed}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")